# Visualization of LLM Output Embeddings

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize
import plotly.graph_objects as go

In [5]:
model_names = [
    "gpt-5.1", "claude_4.5_sonnet", "api-llama3.3", "api-llama4",
    "qwen3-4B", "qwen3-30B",
    "huatuo-7B", "huatuo-8B"
]

In [ ]:
# EXTRACTED only
framing_embeddings_dict = {}
for model in model_names:
    print(f"=== {model} ===")
    # Load and print results for each model (adjust path as needed)
    results = np.load(f"./outputs/evaluation/{model}_embeddings.npz", allow_pickle=True)
    framing_embeddings_dict[model] = results.item()


basic_baseline_embeddings_dict = {}
for model in model_names:
    print(f"=== {model} ===")
    # Load and print results for each model (adjust path as needed)
    results = np.load(f"./outputs/baseline_evaluation/{model}_embeddings.npz", allow_pickle=True)
    basic_baseline_embeddings_dict[model] = results.item()

paraphrased_baseline_embeddings_dict = {}
for model in model_names:
    print(f"=== {model} ===")
    # Load and print results for each model (adjust path as needed)
    results = np.load(f"./outputs/paraphrased_baseline_evaluation/{model}_embeddings.npz", allow_pickle=True)
    paraphrased_baseline_embeddings_dict[model] = results.item()

## Matrix of Embeddings


TODO!

- cosine of positive question answers across models
- cosine of negative question answers across models


## All-Model t-SNE: Grouped by Question Version

Combines response embeddings from **all models** across four question versions (`positive`, `negative`, `second_positive`, `paraphrased_positive`) and visualises the 2-D t-SNE layout coloured by version. Points are drawn from the framing experiment (positive / negative / second_positive) and the paraphrased-baseline experiment (paraphrased_positive).

In [ ]:
## ── helpers ──────────────────────────────────────────────────────────────────

TARGET_VERSIONS = {"positive", "negative", "second_positive", "paraphrased_positive"}

VERSION_COLORS = {
    "positive":             "#4CAF50",   # green
    "negative":             "#F44336",   # red
    "second_positive":      "#FF9800",   # orange
    "paraphrased_positive": "#9C27B0",   # purple
}

MODEL_MARKERS = {
    "gpt-5.1":          "o",
    "claude_4.5_sonnet":"s",
    "api-llama3.3":     "^",
    "api-llama4":       "D",
    "qwen3-4B":         "v",
    "qwen3-30B":        "P",
    "huatuo-7B":        "*",
    "huatuo-8B":        "X",
}

PLOTLY_MODEL_SYMBOLS = {
    "gpt-5.1":          "circle",
    "claude_4.5_sonnet":"square",
    "api-llama3.3":     "triangle-up",
    "api-llama4":       "diamond",
    "qwen3-4B":         "triangle-down",
    "qwen3-30B":        "cross",
    "huatuo-7B":        "star",
    "huatuo-8B":        "x",
}


def parse_key(key):
    """Split 'ReviewID_QuestionCategory_QuestionVersion'.

    QuestionVersion may itself contain underscores (e.g. 'paraphrased_positive'),
    so we split into at most 3 parts.
    """
    parts = key.split("_", 2)
    if len(parts) < 3:
        return None, None, None
    return parts[0], parts[1], parts[2]


## ── collect embeddings ───────────────────────────────────────────────────────

all_vectors    = []
all_versions   = []
all_model_tags = []
all_review_ids = []
all_categories = []

# positive / negative come from the framing experiment
for model, emb_dict in framing_embeddings_dict.items():
    for key, vec in emb_dict.items():
        review_id, category, version = parse_key(key)
        if version in TARGET_VERSIONS:
            all_vectors.append(np.array(vec, dtype=np.float32))
            all_versions.append(version)
            all_model_tags.append(model)
            all_review_ids.append(review_id)
            all_categories.append(category)

# second_positive come from the basic baseline experiment
for model, emb_dict in basic_baseline_embeddings_dict.items():
    for key, vec in emb_dict.items():
        review_id, category, version = parse_key(key)
        if version == "positive2":
            version = "second_positive"
        if version in TARGET_VERSIONS:
            all_vectors.append(np.array(vec, dtype=np.float32))
            all_versions.append(version)
            all_model_tags.append(model)
            all_review_ids.append(review_id)
            all_categories.append(category)

# paraphrased_positive comes from the paraphrased-baseline experiment
for model, emb_dict in paraphrased_baseline_embeddings_dict.items():
    for key, vec in emb_dict.items():
        review_id, category, version = parse_key(key)
        if version in TARGET_VERSIONS:
            all_vectors.append(np.array(vec, dtype=np.float32))
            all_versions.append(version)
            all_model_tags.append(model)
            all_review_ids.append(review_id)
            all_categories.append(category)

X_all        = np.vstack(all_vectors)                    # (N, D)
X_all_normed = normalize(X_all, norm="l2")               # L2-normalise

version_counts = {v: all_versions.count(v) for v in TARGET_VERSIONS}
print(f"Total samples  : {len(all_versions)}")
print(f"Embedding dim  : {X_all.shape[1]}")
print("Samples per version:")
for v, c in version_counts.items():
    print(f"  {v:<25} {c}")

### PCA Pre-reduction

In [ ]:
N_TOTAL  = X_all_normed.shape[0]
N_FEAT   = X_all_normed.shape[1]

# Keep ≤50 components
# Change to e.g. 0.95 to retain 95 % of variance adaptively.
PCA_COMPONENTS = min(50, N_FEAT, N_TOTAL - 1)

pca_all = PCA(n_components=PCA_COMPONENTS, random_state=42)
X_pca   = pca_all.fit_transform(X_all_normed)

var_explained = pca_all.explained_variance_ratio_.sum()
print(f"PCA: {X_all_normed.shape} → {X_pca.shape}")
print(f"Variance explained by {X_pca.shape[1]} components: {var_explained:.1%}")

# ── Scree plot ────────────────────────────────────────────────────────────────
fig_scree, ax_scree = plt.subplots(figsize=(7, 3))
fig_scree.patch.set_facecolor("#0f0f0f")
ax_scree.set_facecolor("#1a1a1a")
ax_scree.plot(
    range(1, len(pca_all.explained_variance_ratio_) + 1),
    pca_all.explained_variance_ratio_.cumsum(),
    color="#4CAF50", linewidth=2,
)
ax_scree.axhline(0.95, color="#F44336", linestyle="--", linewidth=1, label="95 % threshold")
ax_scree.set_xlabel("Number of PCA components", color="#888")
ax_scree.set_ylabel("Cumulative variance explained", color="#888")
ax_scree.set_title("PCA Scree – All-Model Embeddings", color="white")
ax_scree.tick_params(colors="#555")
for spine in ax_scree.spines.values():
    spine.set_edgecolor("#333")
ax_scree.legend(labelcolor="white", facecolor="#222", edgecolor="#444")
plt.tight_layout()
plt.show()

### t-SNE

In [ ]:
perplexity = min(30, N_TOTAL - 1)   # scikit-learn default; scales down for small datasets

tsne_all = TSNE(
    n_components=2,
    perplexity=perplexity,
    n_iter=1000,
    metric="euclidean",   # correct after PCA pre-reduction
    init="pca",
    random_state=42,
    verbose=1,
)

print(f"Running t-SNE on {N_TOTAL} samples "
      f"({X_pca.shape[1]}-dim PCA input, perplexity={perplexity}) …")
coords = tsne_all.fit_transform(X_pca)
print("Done.")

versions_arr = np.array(all_versions)
models_arr   = np.array(all_model_tags)

### Matplotlib — coloured by question version, shaped by model

In [ ]:
VERSION_ORDER = ["positive", "negative", "second_positive", "paraphrased_positive"]
MODEL_ORDER   = [m for m in MODEL_MARKERS if m in set(all_model_tags)]  # only loaded models

fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor("#0f0f0f")
ax.set_facecolor("#1a1a1a")

# Draw one scatter call per (version, model) combination so every combination
# can appear independently in the legend if needed. We build two separate
# legend handles: one for colours (version) and one for shapes (model).
for version in VERSION_ORDER:
    for model in MODEL_ORDER:
        mask = (versions_arr == version) & (models_arr == model)
        if not mask.any():
            continue
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            c=VERSION_COLORS[version],
            marker=MODEL_MARKERS.get(model, "o"),
            s=60,
            alpha=0.75,
            edgecolors="white",
            linewidths=0.3,
            zorder=3,
        )

# ── legend: version colours ───────────────────────────────────────────────────
version_handles = [
    plt.Line2D([0], [0], marker="o", color="none",
               markerfacecolor=VERSION_COLORS[v], markeredgecolor="white",
               markeredgewidth=0.5, markersize=9, label=v)
    for v in VERSION_ORDER
]
leg1 = ax.legend(
    handles=version_handles,
    title="Question version",
    title_fontsize=9,
    loc="upper left",
    framealpha=0.25,
    labelcolor="white",
    facecolor="#222",
    edgecolor="#444",
)
ax.add_artist(leg1)

# ── legend: model shapes ──────────────────────────────────────────────────────
model_handles = [
    plt.Line2D([0], [0], marker=MODEL_MARKERS.get(m, "o"), color="none",
               markerfacecolor="#aaaaaa", markeredgecolor="white",
               markeredgewidth=0.5, markersize=9, label=m)
    for m in MODEL_ORDER
]
ax.legend(
    handles=model_handles,
    title="Model",
    title_fontsize=9,
    loc="upper right",
    framealpha=0.25,
    labelcolor="white",
    facecolor="#222",
    edgecolor="#444",
)

ax.set_title("t-SNE - All Models · Coloured by Question Version",
             color="white", fontsize=14, pad=14)
ax.set_xlabel("t-SNE dim 1", color="#888")
ax.set_ylabel("t-SNE dim 2", color="#888")
ax.tick_params(colors="#555")
for spine in ax.spines.values():
    spine.set_edgecolor("#333")

plt.tight_layout()
plt.savefig("./outputs/figures_all_baselines/tsne_all_models_by_version.pdf", bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("Saved: tsne_all_models_by_version.pdf")

### Matplotlib — faceted by model (small multiples)

In [ ]:
n_models = len(MODEL_ORDER)
n_cols   = 4
n_rows   = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(n_cols * 5, n_rows * 4),
                          sharex=True, sharey=True)
fig.patch.set_facecolor("#0f0f0f")
axes_flat = axes.flatten() if n_models > 1 else [axes]

for ax_idx, model in enumerate(MODEL_ORDER):
    ax = axes_flat[ax_idx]
    ax.set_facecolor("#1a1a1a")
    ax.set_title(model, color="white", fontsize=9, pad=6)
    ax.tick_params(colors="#555", labelsize=6)
    for spine in ax.spines.values():
        spine.set_edgecolor("#333")

    model_mask = models_arr == model
    # grey background: all other points for spatial reference
    ax.scatter(coords[~model_mask, 0], coords[~model_mask, 1],
               c="#444444", s=10, alpha=0.25, zorder=1, rasterized=True)

    for version in VERSION_ORDER:
        mask = model_mask & (versions_arr == version)
        if not mask.any():
            continue
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            c=VERSION_COLORS[version],
            s=40,
            alpha=0.85,
            edgecolors="white",
            linewidths=0.3,
            label=version,
            zorder=3,
        )

# Hide unused subplots
for ax in axes_flat[n_models:]:
    ax.set_visible(False)

# Shared legend below the grid
version_handles = [
    plt.Line2D([0], [0], marker="o", color="none",
               markerfacecolor=VERSION_COLORS[v], markeredgecolor="white",
               markeredgewidth=0.5, markersize=9, label=v)
    for v in VERSION_ORDER
]
fig.legend(
    handles=version_handles,
    loc="lower center",
    ncol=len(VERSION_ORDER),
    framealpha=0.25,
    labelcolor="white",
    facecolor="#222",
    edgecolor="#444",
    fontsize=9,
    bbox_to_anchor=(0.5, -0.02),
)

fig.suptitle("t-SNE by Model — Coloured by Question Version",
             color="white", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("./outputs/figures_all_baselines/tsne_faceted_by_model.pdf", bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("Saved: tsne_faceted_by_model.pdf")

### Plotly — interactive (hover for metadata, toggle by version × model)

In [ ]:
traces = []

for version in VERSION_ORDER:
    for model in MODEL_ORDER:
        mask = (versions_arr == version) & (models_arr == model)
        if not mask.any():
            continue

        idx = np.where(mask)[0]
        hover = [
            f"<b>version:</b> {all_versions[i]}<br>"
            f"<b>model:</b>   {all_model_tags[i]}<br>"
            f"<b>review:</b>  {all_review_ids[i]}<br>"
            f"<b>category:</b> {all_categories[i]}"
            for i in idx
        ]

        traces.append(go.Scatter(
            x=coords[mask, 0],
            y=coords[mask, 1],
            mode="markers",
            name=f"{version} | {model}",
            legendgroup=version,          # clicking version legend hides all its models
            legendgrouptitle_text=version if model == MODEL_ORDER[0] else None,
            marker=dict(
                size=8,
                color=VERSION_COLORS[version],
                symbol=PLOTLY_MODEL_SYMBOLS.get(model, "circle"),
                opacity=0.82,
                line=dict(width=0.6, color="white"),
            ),
            text=hover,
            hovertemplate="%{text}<extra></extra>",
        ))

fig_plotly = go.Figure(
    data=traces,
    layout=go.Layout(
        title="t-SNE - All Models · Question Version (interactive)",
        xaxis_title="t-SNE dim 1",
        yaxis_title="t-SNE dim 2",
        plot_bgcolor="#1a1a1a",
        paper_bgcolor="#0f0f0f",
        font=dict(color="white"),
        legend=dict(
            bgcolor="#1e1e1e",
            bordercolor="#444",
            groupclick="toggleitem",   # click legend item → hide just that trace
        ),
        width=1100,
        height=700,
    ),
)

fig_plotly.write_html("./outputs/figures_all_baselines/tsne_all_models_interactive.html")
fig_plotly.show()
print("Saved: tsne_all_models_interactive.html")